# Agent Deep Dive — LangChain Orchestration for Pharma MMM
## How the Multi-Agent System Works Under the Hood

> **What this notebook covers:**  
> A detailed walkthrough of the LangChain agent architecture — how the planner, analytics,  
> and insight agents are wired together, how tools are called, and how to extend the system.
>
> **Prerequisites:**  
> Complete `01_mmm_pipeline_walkthrough.ipynb` first.

---

### Architecture Overview

```
User Query (natural language)
        │
        ▼
┌───────────────────┐
│   Planner Agent   │  ← Orchestrator: decomposes task, routes to sub-agents
└────────┬──────────┘
         │
    ┌────┴─────┐
    ▼          ▼
┌──────────┐  ┌──────────────┐
│Analytics │  │ Insight      │
│Agent     │  │ Agent        │  ← LLM converts numbers → pharma narrative
└────┬─────┘  └──────────────┘
     │
     ▼
LangChain Tools (Python functions wrapped as tools)
  ├── apply_all_transforms_tool  (adstock + saturation)
  ├── run_ols_mmm_tool           (Ridge MMM model)
  └── run_budget_optimizer_tool  (scipy optimisation)
```

In [1]:
import sys, os

# ── Path resolution ───────────────────────────────────────────────────────────
# Works whether you launch Jupyter from the project root OR the notebooks/ folder
_here = os.path.abspath('')
PROJECT_ROOT = _here if os.path.exists(os.path.join(_here, 'config')) else os.path.dirname(_here)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.getcwd().endswith('notebooks') else os.getcwd())

import json
import numpy as np
import pandas as pd
import yaml
import warnings
warnings.filterwarnings('ignore')

print('✓ Setup complete')

Project root: /Users/shabam/Projects/pharma-mmm-agent
✓ Setup complete


---
## 1. What is a LangChain Tool?

A **LangChain tool** is a Python function that an LLM agent can call autonomously.  
You decorate any function with `@tool` and LangChain handles:
- Exposing it to the LLM with a name and description
- Parsing the LLM's JSON argument calls into Python
- Returning the result back to the agent's reasoning loop

**Key design principle:** Tools should be **narrow and composable** — each tool does one thing well.  
The agent decides which tools to call, in what order, and how to combine their outputs.

In [2]:
from langchain.tools import tool

@tool
def example_tool(channel_name: str, value: float) -> str:
    """
    This docstring is what the LLM reads to decide when to call this tool.
    Be specific: describe WHAT the tool does, WHEN to use it, and what it returns.

    Args:
        channel_name : the marketing channel to process
        value        : a numerical parameter
    Returns:
        A string result the agent can read and reason about
    """
    return f'Processed {channel_name} with value {value}'

# Inspect what the LLM sees
print('Tool name:        ', example_tool.name)
print('Tool description: ', example_tool.description[:120])
print('Tool args schema: ', example_tool.args)

Tool name:         example_tool
Tool description:  This docstring is what the LLM reads to decide when to call this tool.
Be specific: describe WHAT the tool does, WHEN to
Tool args schema:  {'channel_name': {'title': 'Channel Name', 'type': 'string'}, 'value': {'title': 'Value', 'type': 'number'}}


In [3]:
# Inspect our actual MMM tools — what the agent sees when deciding which tool to call
from tools.transforms import apply_all_transforms_tool
from tools.ols_mmm_tool import run_ols_mmm_tool
from tools.optimizer_tool import run_budget_optimizer_tool

mmm_tools = [apply_all_transforms_tool, run_ols_mmm_tool, run_budget_optimizer_tool]

for t in mmm_tools:
    print(f'\n{"-"*60}')
    print(f'TOOL: {t.name}')
    print(f'ARGS: {list(t.args.keys())}')
    print(f'DESC: {t.description[:200]}...')


------------------------------------------------------------
TOOL: apply_all_transforms_tool
ARGS: ['data_path', 'config_path']
DESC: Apply adstock and saturation transforms to ALL channels at once,
using decay and saturation parameters from config.yaml.

Use this as the first analytical step — transforms all 12 channels
in one call...

------------------------------------------------------------
TOOL: run_ols_mmm_tool
ARGS: ['data_path', 'config_path', 'freq']
DESC: Run a Ridge-regularised Marketing Mix Model on the transformed pharma dataset.
Ridge regression prevents multicollinearity-driven sign flips on correlated
pharma channels — standard MMM practice.

Use...

------------------------------------------------------------
TOOL: run_budget_optimizer_tool
ARGS: ['results_path', 'config_path', 'total_budget_k', 'freq']
DESC: Optimise budget allocation using fitted ROI from the Ridge MMM results.

Args:
    results_path   : path to _ols_results.json
    config_path    : path to conf

---
## 2. Calling Tools Directly (Without the LLM)

You can invoke any tool directly using `.invoke({...})` — no LLM needed.  
This is useful for testing, debugging, and running the pipeline without an API key.

In [4]:
# Step 1: Transforms tool
print('Calling apply_all_transforms_tool directly...\n')
r1 = apply_all_transforms_tool.invoke({
    'data_path':   'data/raw/mmm_weekly.csv',
    'config_path': 'config/config.yaml'
})
print(r1)

Calling apply_all_transforms_tool directly...

All channel transforms complete (12 channels):
  rep_visits                decay=0.6  sat=0.55  avg_saturated=0.701
  medical_congress          decay=0.75  sat=0.45  avg_saturated=0.802
  journal_advertising       decay=0.5  sat=0.65  avg_saturated=0.807
  hcp_email                 decay=0.35  sat=0.8  avg_saturated=0.736
  hcp_digital               decay=0.4  sat=0.7  avg_saturated=0.698
  speaker_programs          decay=0.7  sat=0.5  avg_saturated=0.847
  samples_coupons           decay=0.55  sat=0.6  avg_saturated=0.807
  dtc_tv                    decay=0.5  sat=0.75  avg_saturated=0.691
  dtc_digital               decay=0.38  sat=0.72  avg_saturated=0.695
  dtc_ooh                   decay=0.45  sat=0.68  avg_saturated=0.646
  patient_email             decay=0.3  sat=0.82  avg_saturated=0.761
  patient_advocacy          decay=0.65  sat=0.55  avg_saturated=0.757

Transformed dataset saved to: data/raw/mmm_weekly_transformed.csv


In [5]:
# Step 2: Ridge MMM tool
# Note: reads the _transformed.csv produced by the previous tool
# This is the key pattern — tools build on each other's outputs
print('Calling run_ols_mmm_tool directly...\n')
r2 = run_ols_mmm_tool.invoke({
    'data_path':   'data/raw/mmm_weekly_transformed.csv',
    'config_path': 'config/config.yaml',
    'freq':        'weekly'
})
print(r2)

Calling run_ols_mmm_tool directly...

Ridge MMM Results (R²=0.928, MAPE=4.2%)
Observations: 104 weekly periods
Baseline scripts: 11,173

Channel                        Type  Spend $K     ROI      Contribution%
------------------------------------------------------------------------
Out-of-Home Advertising        dtc   $5,160      0.210    25.0%
Patient Advocacy Partnerships  dtc   $2,272      0.490    21.0%
Speaker Bureau Programs        hcp   $4,244      0.630    19.2%
DTC Television                 dtc   $25,536     0.280    10.8%
DTC Digital & Social           dtc   $9,798      0.350    8.9%
HCP Programmatic Digital       hcp   $6,074      0.392    5.4%
Journal Advertising            hcp   $4,285      0.308    5.1%
HCP Email Campaigns            hcp   $1,712      0.252    4.8%
Field Rep Visits               hcp   $21,554     0.228    0.0%
Medical Congress & Symposia    hcp   $7,630      0.312    0.0%
Samples & Co-pay Coupons       hcp   $9,872      0.198    0.0%
Patient CRM Email   

In [6]:
# Step 3: Budget optimiser tool
with open('data/raw/mmm_weekly_ols_results.json') as f:
    ols_data = json.load(f)

total_spend = sum(ch['total_spend_k'] for ch in ols_data['channels'].values())
avg_weekly  = round(total_spend / ols_data['n_observations'], 1)
print(f'Average weekly budget: ${avg_weekly}K')

r3 = run_budget_optimizer_tool.invoke({
    'results_path':   'data/raw/mmm_weekly_ols_results.json',
    'config_path':    'config/config.yaml',
    'total_budget_k': avg_weekly,
    'freq':           'weekly'
})
print(r3)

Average weekly budget: $956.2K
Error in budget optimiser: 'base'


---
## 3. The Agent Reasoning Loop (ReAct Pattern)

When running with a real API key, the LLM follows a **Reason + Act** loop:

```
1. USER INPUT  →  "Run full MMM analysis on mmm_weekly.csv"
2. LLM THINKS  →  "I need to: (a) apply transforms, (b) fit model, (c) optimise"
3. TOOL CALL   →  apply_all_transforms_tool(...)
   TOOL RESULT →  "All channel transforms complete..."
4. LLM THINKS  →  "Transforms done. Now run the MMM model."
5. TOOL CALL   →  run_ols_mmm_tool(...)
   TOOL RESULT →  "Ridge MMM Results (R²=0.928)..."
6. LLM THINKS  →  "Model fitted. Extract budget and run optimiser."
7. TOOL CALL   →  run_budget_optimizer_tool(...)
   TOOL RESULT →  "Budget Optimisation — +8.5% projected uplift..."
8. LLM THINKS  →  "All steps complete. Summarise."
9. FINAL OUTPUT → Structured summary for commercial team
```

**The LLM is the orchestrator — it decides the sequence.**  
**The tools are the executors — they do the actual computation.**

In [7]:
# Simulate the ReAct loop (without any API call)
class MockAgentLoop:
    """Simulates the LangChain ReAct loop. Shows exact sequence of tool calls."""
    def __init__(self, tools):
        self.tools = {t.name: t for t in tools}

    def think(self, thought):
        print(f'\n🤔 AGENT THINKS: {thought}')

    def act(self, tool_name, args):
        print(f'🔧 CALLS TOOL:   {tool_name}({list(args.keys())})')
        result = self.tools[tool_name].invoke(args)
        print(f'📋 TOOL RESULT:  {result[:120].replace(chr(10), " ")}...')
        return result

agent = MockAgentLoop(mmm_tools)

agent.think("First step: apply transforms to prepare the feature matrix.")
agent.act('apply_all_transforms_tool', {'data_path': 'data/raw/mmm_weekly.csv', 'config_path': 'config/config.yaml'})

agent.think("Transforms done. Fit Ridge MMM on the transformed data.")
agent.act('run_ols_mmm_tool', {'data_path': 'data/raw/mmm_weekly_transformed.csv', 'config_path': 'config/config.yaml', 'freq': 'weekly'})

agent.think("Model fitted. Run budget optimiser using fitted ROI values.")
agent.act('run_budget_optimizer_tool', {'results_path': 'data/raw/mmm_weekly_ols_results.json', 'config_path': 'config/config.yaml', 'total_budget_k': avg_weekly, 'freq': 'weekly'})

agent.think("All steps complete. Generate final summary for the commercial team.")
print('\n✅ Agent pipeline complete — 3 tool calls, 0 API requests')


🤔 AGENT THINKS: First step: apply transforms to prepare the feature matrix.
🔧 CALLS TOOL:   apply_all_transforms_tool(['data_path', 'config_path'])
📋 TOOL RESULT:  All channel transforms complete (12 channels):   rep_visits                decay=0.6  sat=0.55  avg_saturated=0.701   me...

🤔 AGENT THINKS: Transforms done. Fit Ridge MMM on the transformed data.
🔧 CALLS TOOL:   run_ols_mmm_tool(['data_path', 'config_path', 'freq'])
📋 TOOL RESULT:  Ridge MMM Results (R²=0.928, MAPE=4.2%) Observations: 104 weekly periods Baseline scripts: 11,173  Channel              ...

🤔 AGENT THINKS: Model fitted. Run budget optimiser using fitted ROI values.
🔧 CALLS TOOL:   run_budget_optimizer_tool(['results_path', 'config_path', 'total_budget_k', 'freq'])
📋 TOOL RESULT:  Error in budget optimiser: 'base'...

🤔 AGENT THINKS: All steps complete. Generate final summary for the commercial team.

✅ Agent pipeline complete — 3 tool calls, 0 API requests


---
## 4. Running With a Real LLM (Requires API Key)

Once you add your OpenAI key to `.env`, the agent runs fully autonomously.

In [8]:
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY', '')
has_key = bool(api_key) and 'your-' not in api_key and len(api_key) > 20

if has_key:
    print('✅ API key found — run the next cell for full LLM-orchestrated pipeline')
else:
    print('ℹ️  No API key yet — add to .env: OPENAI_API_KEY=sk-...')
    print('   Cost for one full run: ~$0.05 (GPT-4o)')

ℹ️  No API key yet — add to .env: OPENAI_API_KEY=sk-...
   Cost for one full run: ~$0.05 (GPT-4o)


In [9]:
# Uncomment after adding your API key to .env
# from agents.analytics_agent import run_analytics_pipeline
#
# output = run_analytics_pipeline(
#     data_path='data/raw/mmm_weekly.csv',
#     config_path='config/config.yaml',
#     freq='weekly',
#     verbose=True  # shows agent reasoning steps live
# )
# print(output)

---
## 5. How to Add Your Own Tool

Writing a new tool follows a simple 3-step pattern.

In [10]:
# Example: spend anomaly detection tool
@tool
def detect_spend_anomalies_tool(data_path: str, threshold_sigma: float = 2.0) -> str:
    """
    Detect anomalous spend weeks across all channels using z-score analysis.
    Use before model fitting to flag data quality issues.

    Args:
        data_path        : path to MMM CSV dataset
        threshold_sigma  : z-score threshold (default 2.0)
    Returns:
        Summary of anomalous weeks detected
    """
    df = pd.read_csv(data_path, parse_dates=['date'])
    with open('config/config.yaml') as f:
        cfg = yaml.safe_load(f)

    anomalies = []
    for ch, params in cfg['channels'].items():
        if ch not in df.columns: continue
        z = (df[ch] - df[ch].mean()) / (df[ch].std() + 1e-9)
        for idx in df[abs(z) > threshold_sigma].index:
            anomalies.append({
                'date': str(df.loc[idx,'date'].date()),
                'channel': params['label'],
                'spend_k': round(df.loc[idx, ch], 1),
                'z_score': round(z[idx], 2)
            })

    if not anomalies:
        return f'No anomalies detected at {threshold_sigma}σ — data quality looks clean.'

    lines = [f'{len(anomalies)} anomalous spend weeks detected:']
    for a in sorted(anomalies, key=lambda x: -abs(x['z_score']))[:8]:
        lines.append(f"  {a['date']}  {a['channel']:<28} ${a['spend_k']:>7.1f}K  z={a['z_score']}")
    return '\n'.join(lines)

# Test it
print(detect_spend_anomalies_tool.invoke({'data_path': 'data/raw/mmm_weekly.csv'}))

54 anomalous spend weeks detected:
  2023-01-16  HCP Programmatic Digital     $  130.8K  z=5.88
  2023-02-20  Medical Congress & Symposia  $  206.9K  z=5.51
  2022-11-14  Out-of-Home Advertising      $  121.1K  z=5.47
  2023-11-06  Patient Advocacy Partnerships $   56.8K  z=5.11
  2022-02-14  Field Rep Visits             $  539.0K  z=4.78
  2022-07-04  Samples & Co-pay Coupons     $  183.0K  z=4.52
  2022-12-26  DTC Television               $  540.5K  z=4.24
  2022-05-23  Field Rep Visits             $  489.7K  z=4.07


In [11]:
# To register this with the analytics agent, add it to tools list in analytics_agent.py:
#
#   from tools.anomaly_tool import detect_spend_anomalies_tool
#
#   tools = [
#       apply_all_transforms_tool,
#       run_ols_mmm_tool,
#       run_budget_optimizer_tool,
#       detect_spend_anomalies_tool,   # ← add here
#   ]
#
# The agent discovers it automatically from the docstring — no other changes needed.

print('3-step pattern to add any new tool:')
print('  1. Write function with @tool decorator')
print('  2. Write a clear, specific docstring — this is what the LLM reads')
print('  3. Import and add to tools list in analytics_agent.py')

3-step pattern to add any new tool:
  1. Write function with @tool decorator
  2. Write a clear, specific docstring — this is what the LLM reads
  3. Import and add to tools list in analytics_agent.py


---
## Summary

### What you learned
- ✅ How `@tool` works — function + docstring = agent-callable tool
- ✅ How to invoke tools directly for testing (no API cost)
- ✅ How the ReAct reasoning loop works — think → act → observe → repeat
- ✅ How to write and register custom tools

### Key files to study
| File | What to read |
|---|---|
| `tools/transforms.py` | The `@tool` decorator pattern |
| `agents/analytics_agent.py` | Tool registration + system prompt |
| `agents/planner_agent.py` | Orchestrator coordinating sub-agents |
| `agents/insight_agent.py` | LLM converting model output to narrative |
| `config/config.yaml` | All model parameters |

---
*Built by Shubham Kumar — Senior Data Scientist | shubham.mle@gmail.com*